In [1]:
from decouple import config
from sqlalchemy import create_engine
import pandas as pd
from datetime import datetime, timedelta
import time 
from datetime import datetime
from sqlalchemy import text
import oracledb
import sys

In [2]:
oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = 'User_oper'
pw = 'TmLQL$Yq.1'
hostname='10.56.1.76'
service_name="WNET"
port = 1527

engine0 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)
connection0 = engine0.connect()

In [3]:
prog_asis_mi_consulta = pd.read_sql_query(f"""select  
ca.redasiscod                                                              AS CODRED,
(SELECT ra.redasisdes FROM cmras10 ra WHERE ra.redasiscod = ca.redasiscod) AS RED,
a.cenasicod                                                                as CODCENTRO, --xls
ca.cenasidescor                                                            AS CENTRO,
to_char(a.properfec,'dd/mm/yyyy')                                          as FECHA_PROGRAMACION,
(decode(a.tipohorprogcod,'1','NORMAL','2','EXTRAS','3','RPCT','4','EXTRAS+CITAS','5','COMPENSACION')) as TIP_PROGRAMACION,
(select h.arehosdes  from cmaho10  h where  a.arehoscod  = h.arehoscod)    as AREA,
(select e.servhoscod from cmsho10  e where  a.servhoscod = e.servhoscod)   as CODSERVICIO,
(select e.servhosdes from cmsho10  e where  a.servhoscod = e.servhoscod)   as SERVICIO,
(select i.actcod     from cmact10  i where  a.actcod     = i.actcod)             as CODACTIVIDAD,
(select i.actdes     from cmact10  i where  a.actcod     = i.actcod)             as ACTIVIDAD,
(select j.actespcod  from cmace10  j where  a.actcod     = j.actcod 
and a.actespcod = j.actespcod)                                                   as CODSUBACTIVIDAD,
(select j.actespnom  from cmace10  j where  a.actcod     = j.actcod 
and a.actespcod = j.actespcod)                                                   as SUBACTIVIDAD,
(select x1.estprogcitnom from cbepc10 x1 where x1.estprogcitcod = a.estprogcitcod) as ESTADO_PROGRAMACION,
(SELECT mt.motsusprogdes FROM cbmsp10 mt WHERE mt.motsusprogcod = a.motsusprogcod) AS MOTIVO_SUSPENSION
from ctppe10 a
LEFT OUTER join ctpco10 c on a.oricenasicod      = c.proconoricenasicod
                            and a.cenasicod        = c.proconcenasicod
                            and a.arehoscod        = c.proconarehoscod
                            and a.servhoscod       = c.proconservhoscod
                            and a.actcod           = c.proconactcod
                            and a.actespcod        = c.proconactespcod
                            and a.tipdocidenpercod = c.procontipdocidenpercod
                            and a.perasisdocidennum  = c.proconperasisdocidennum
                            and a.properfec          = c.proconfec
                            and a.properturhorini    = c.proconturhorini
                            and a.properturhorfin    = c.proconturhorfin
/*left outer join cmprs10 m on m.tipdocidenpercod = a.tipdocidenpercod
                         and m.perasisdocidennum = a.perasisdocidennum*/
LEFT OUTER JOIN cmcas10 ca ON ca.oricenasicod = a.oricenasicod
                           AND ca.cenasicod   = a.cenasicod                   
where  
a.estprogcitcod   = '2'   -- Preguntar si todos los estados de la programacion o solo las aprobadas .....???
and a.properfec >= TO_DATE('20240101', 'yyyymmdd')
and a.properfec < TO_DATE('20240830', 'yyyymmdd') + 1
--and (a.estprogcitcod   = '2' OR a.estprogcitcod = '3')  -- Aprob/Bloq upd 010219
and a.properestregcod = '1'""", con=connection0)

In [4]:
DB_USER='postgres'
DB_PASSWORD='AKindOfMagic'
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST='10.0.1.228'
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()

In [5]:
prog_asis_mi_consulta.to_sql(name=f'prog_asis_mi_consulta', con=connection2, if_exists='replace', index=False,chunksize=10000)

818186

In [6]:
fecha = pd.read_sql_query("SELECT fec_ini FROM etl_act where id_mod=21.0", con=connection2)
fecha= fecha.iloc[0, 0]
print(fecha)

now = datetime.now()

query=f"UPDATE etl_act SET fec_act ='{now}' WHERE id_mod=21"

c1= text(query)
connection2.execute(c1)

None


In [7]:
connection0.close()
connection2.close()
engine0.dispose()
engine2.dispose()